In [0]:
%run ../includes


In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# 1. Read bronze source
bronze_source = spark.table(f"{catalog}.bronze.customers")

# 2. Drop meta columns to match silver schema
bronze_source = bronze_source.drop("file_name", "file_path", "ingestion_date")

# 3. Deduplicate: keep latest record per customer_id (by updated_at)
dedup_window = Window.partitionBy("customer_id").orderBy(col("updated_at").desc_nulls_last())
bronze_source = (
    bronze_source
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# 4. Clean: trim all string columns at once (avoids loop lint warning)
string_cols = [f.name for f in bronze_source.schema.fields if f.dataType.simpleString() == "string"]
bronze_source = bronze_source.withColumns({c: trim(col(c)) for c in string_cols})

# 5. Validate email: set invalid to NULL
bronze_source = bronze_source.withColumn(
    "email",
    when(col("email").rlike(r'^[^@\s]+@[^@\s]+\.[^@\s]+$'), col("email"))
    .otherwise(lit(None).cast("string"))
)

# 6. Record count before merge
before_count = spark.table(f"{catalog}.silver.silver_customers").count()

# 7. SCD Type 1 MERGE using DeltaTable API
silver_dt = DeltaTable.forName(spark, f"{catalog}.silver.silver_customers")

(
    silver_dt.alias("target")
    .merge(
        bronze_source.alias("source"),
        col("target.customer_id") == col("source.customer_id")
    )
    .whenMatchedUpdate(
        # Only update if any tracked column has actually changed (null-safe comparison)
        condition=(
            ~ col("target.email").eqNullSafe(col("source.email"))
            | ~ col("target.phone").eqNullSafe(col("source.phone"))
            | ~ col("target.address").eqNullSafe(col("source.address"))
            | ~ col("target.city").eqNullSafe(col("source.city"))
            | ~ col("target.state").eqNullSafe(col("source.state"))
            | ~ col("target.postal_code").eqNullSafe(col("source.postal_code"))
        ),
        set={
            "email":       "source.email",
            "phone":       "source.phone",
            "address":    "source.address",
            "city":        "source.city",
            "state":       "source.state",
            "postal_code": "source.postal_code",
            "updated_at":  "source.updated_at",
        },
    )
    .whenNotMatchedInsertAll()
    .execute()
)